# 2. Train and test (run every time)
Uses the saved data, so nothing is downloaded. Embeddings are cached on Drive: only examples you
added since last time are embedded. To add examples, edit `own_examples.py` in your Drive folder,
then run this notebook from the top (Runtime → Run all).

In [ ]:
# Connect your Drive and load the shared code
from google.colab import drive
drive.mount("/content/drive")
import sys, importlib
sys.path.insert(0, "/content/drive/MyDrive/kq-scope-forest")
import common; importlib.reload(common)

In [ ]:
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import own_examples; importlib.reload(own_examples)

## Build the training set

In [ ]:
test = pd.read_csv(f"{common.DATA}/test.csv")
public = pd.read_csv(f"{common.DATA}/public.csv")
own = pd.DataFrame([(t, "in_scope", "own") for t in own_examples.own_in]
                   + [(t, "off_topic", "own") for t in own_examples.own_off], columns=["text", "label", "source"])
train = pd.concat([public, own])
train["text"] = train["text"].str.strip()
train = train.drop_duplicates("text")
train = train[~train["text"].str.lower().isin(set(test["text"].str.lower().str.strip()))]
train.to_csv(f"{common.DATA}/train.csv", index=False)
print(train["label"].value_counts())

## Turn text into numbers (cached)

In [ ]:
X_train = common.embed(train["text"])
X_test = common.embed(test["text"])

## Train the Random Forest

In [ ]:
model = RandomForestClassifier(n_estimators=300, max_depth=None, min_samples_leaf=2,
                               class_weight="balanced", random_state=42, n_jobs=-1)
model.fit(X_train, train["label"])
joblib.dump(model, f"{common.MODELS}/forest.joblib")

## Results on the held-out questions

In [ ]:
pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(test["label"], pred), 3))
print(classification_report(test["label"], pred, digits=3))
print(confusion_matrix(test["label"], pred, labels=["in_scope", "off_topic"]))

In [ ]:
# All mistakes
wrong = test[pred != test["label"]]
for t, l in zip(wrong["text"], wrong["label"]):
    print(l, "|", t)

## The "not sure" band

In [ ]:
p_off = model.predict_proba(X_test)[:, list(model.classes_).index("off_topic")]
for low, high in [(0.4, 0.6), (0.3, 0.7), (0.2, 0.8)]:
    sure = (p_off < low) | (p_off > high)
    sure_pred = ["off_topic" if p > high else "in_scope" for p in p_off[sure]]
    acc = accuracy_score(test["label"][sure], sure_pred)
    print(f"band {low}-{high}: forest decides {sure.mean():.0%} alone, accuracy on those {acc:.1%}")

In [ ]:
# Mistakes the forest makes even when it is confident (these matter most)
low, high = 0.3, 0.7
sure = (p_off < low) | (p_off > high)
band_pred = pd.Series(["off_topic" if p > high else "in_scope" for p in p_off], index=test.index)
bad = test[sure & (band_pred != test["label"])]
for t, l in zip(bad["text"], bad["label"]):
    print(l, "|", t)